# 03B — Run the training and batch entry points step by step

This notebook acts as the orchestrator: it calls the same `execute(...)` functions
used by the container jobs, supplies local paths instead of `/opt/ml/...`, and
shows the data and artifacts passed between steps. Run it with a fresh kernel.

**Training:** `Prepare (read → split → select features) → Train → Evaluate → QualityGate`

**Batch:** `ValidatePackage (local mode) → Preprocess → LoadModel → score requests → Postprocess`

The final direct-Predict call is a separate convenience path for comparison.
All model computations use the real source and checkpoint. Nothing is mocked.
The batch requests use the serving application's in-process client, without
starting Docker, Gunicorn or an HTTP listener.

The included CSV example needs the kernel's pandas, scikit-learn, joblib, Flask
and selected modeling framework. It does not install packages or call AWS.
If Flask is missing, add the application's `flask==3.1.2` dependency to this kernel
through your environment's package-installation process.

Keep this notebook open alongside the use-case scripts. Continue with **03C**
for real container-based Local execution and **03D** for managed execution.
Container packages can differ from those in the notebook kernel.

| Pipeline action | What executes here | What still needs managed execution |
|---|---|---|
| Prepare | `prepare.execute`: source reader, split and feature selection | Processing job and S3 transfers |
| Train | `train.execute`: fitted preprocessing, framework fit and saved checkpoint | Training job |
| Evaluate | `evaluate.execute`: held-out predictions, metrics and quality decision | Processing job |
| QualityGate / RegisterModel | Read the actual quality result; stop on failure | Native Condition/Fail and registration |
| ValidatePackage | `validate_package.execute(require_approval=False)`: explicit local-development mode | Approval and image/checkpoint checks against the registry |
| Preprocess | `preprocess.execute`: saved feature columns and JSON-lines shards | Processing job |
| CreateModel / Transform | `serve.execute`: load the checkpoint and run its actual request handler | CreateModel API, Batch Transform job and container server |
| Postprocess | `postprocess.execute`: collect raw scores and format predictions | Processing job |

Registry approval, IAM, S3 transfer, container startup, concurrency and instance
limits are not validated by this Python walkthrough.

## Use the current kernel and this checkout

- Locate the extracted repository and import the production entry-point functions.
- Keep the current Python environment and show its package versions and source paths.
- Use the bundled CSV; all checkpoints, reports and intermediate files stay local.

In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import sys
import pandas as pd

# Locate and import the model code from this checkout.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'mlops-models/src/mlops_models').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the extracted repository folder.')
sys.path.insert(0, str(ROOT / 'mlops-models/src'))
from mlops_models.entrypoints import (
    prepare, train, evaluate, validate_package, preprocess, serve, postprocess, predict,
)

# Show the interpreter, dependencies and exact entry-point files used.
packages = []
for name in ('pandas', 'scikit-learn', 'joblib', 'flask', 'autogluon.tabular', 'torch'):
    try:
        installed_version = version(name)
    except PackageNotFoundError:
        installed_version = 'not installed'
    packages.append({'package': name, 'version': installed_version})
print('Python:', sys.version.split()[0])
print('Kernel:', sys.executable)
display(pd.DataFrame(packages))
entrypoints = (prepare, train, evaluate, validate_package, preprocess, serve, postprocess, predict)
display(pd.DataFrame([{'entry point': module.__name__.rsplit('.', 1)[-1],
                       'source file': str(Path(module.__file__).relative_to(ROOT))}
                      for module in entrypoints]))

## Choose the model, inputs and local job directories

- Choose the use case and display its configuration and customization files.
- Select the included CSV and create a new run folder for this execution.
- Use small inference shards so the handoff between multiple batch files is visible.

Framework parameters are overrides. Training resolves the defaults and saves them
with the checkpoint; inference uses that saved configuration. Edit the use-case
Python files and restart the kernel to load your changes.

In [ ]:
from datetime import datetime, timezone
import json, os, shutil

# Select the source, configuration and batch size.
MODEL = os.environ.get('MLOPS_MODEL', 'usecase2')
MODEL_FOLDER = ROOT / 'mlops-models/src/mlops_models/models' / MODEL
config = json.loads((MODEL_FOLDER / 'model-config.json').read_text())
CSV_OPTIONS = {'dtype': {config['identifier']: str}, 'keep_default_na': False,
               'na_values': ['', r'\N']}
SOURCE = ROOT / 'examples/data' / config['example_input']
BATCH_ROWS = 50

# Keep the same job directory layout under one local run folder.
RUN = ROOT / '.runs/notebook-entrypoints' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
PREPARED = RUN / 'prepare/opt/ml/processing/output'
TRAINING = RUN / 'train/opt/ml'
EVALUATION = RUN / 'evaluate/opt/ml/processing/output'
AUTHORIZATION = RUN / 'validate-package/opt/ml/processing/output'
BATCH_INPUT = RUN / 'preprocess/opt/ml/processing/output'
BATCH_SCORES = RUN / 'transform/output'
BATCH_OUTPUT = RUN / 'postprocess/opt/ml/processing/output'
(TRAINING / 'input/data').mkdir(parents=True)
print('Run folder:', RUN)
print('Feature selection:', MODEL_FOLDER / 'training/feature_selection.py')
print('Training preprocessing:', MODEL_FOLDER / 'training/preprocess.py')
print('Evaluation:', MODEL_FOLDER / 'training/evaluate.py')
display(config)
display(pd.read_csv(SOURCE, nrows=5, **CSV_OPTIONS))

## Prepare: read data, split rows and select features

- Call `prepare.execute`, which reads the input and splits it before feature selection.
- Fit the selected use case's feature-selection logic on training rows only.
- Inspect selected features, their importance scores and all three resulting splits.

For usecase2, the call is
`prepare.execute → prepare_training_splits → use_case_read / use_case_split_data / use_case_select_features`.
Its feature-selection file overrides the base implementation. Get Data and feature
selection are functions inside Prepare, not separate managed pipeline steps.

The ranking files below come from the current example selector. If you replace it
with code that writes different reports, update these inspection lines to read
your reports. The selected split CSVs are the handoff to Train and Evaluate.

The included usecase2 example requests `top_k=50` but has 22 candidate features,
so it ranks and keeps all 22. Lower `feature_selection.top_k` in the model config
to inspect an actual reduction, then restart the kernel and rerun.

In [ ]:
# Run the production Prepare entry point.
split_dir = prepare.execute(MODEL, SOURCE, PREPARED)

# Inspect the feature-selection outputs and prepared split dataframes.
selected_features = pd.read_csv(PREPARED / 'selection/selected_features.csv')
feature_importance = pd.read_csv(PREPARED / 'selection/feature_importance.csv')
splits = {name: pd.read_csv(split_dir / name / f'{name}.csv', **CSV_OPTIONS)
          for name in ('train', 'validation', 'test')}
selected = splits['train']
print('Selected features:')
display(selected_features)
print('Feature importance, highest first:')
display(feature_importance)
display(pd.DataFrame([{'split': name, 'rows': len(frame), 'columns': len(frame.columns)}
                      for name, frame in splits.items()]))
display(selected.head())

## Hand the prepared splits to training

- Copy the train and validation CSV folders into the next job's local input directory.
- Keep the test CSV outside training for the evaluation step.
- Show the input directory; managed SageMaker transfers these inputs through S3.

In [ ]:
training_input = TRAINING / 'input/data'
for name in ('train', 'validation'):
    shutil.copytree(split_dir / name, training_input / name)
print('Training inputs:', training_input)
print('The test split stays with Prepare for evaluation.')

## Train: fit preprocessing, train the model and save the checkpoint

- Call `train.execute` with the prepared train/validation folders and output paths.
- Fit the use case's preprocessing on training rows, then call the selected framework.
- Inspect the resolved parameters, checkpoint metadata and exact transformed X/y.

The framework receives `PreparedTrainingData` and returns
`FitResult(predictor, selected_model)`. Training saves the predictor, fitted
preprocessing and configuration, then returns the checkpoint path.
The test split stays outside fitting.

In [ ]:
# Train using the prepared input channels and save the checkpoint.
checkpoint_path = train.execute(MODEL, training_input, TRAINING / 'model', TRAINING / 'output/data')

# Inspect the saved configuration and exact inputs supplied to the framework.
metadata = json.loads((checkpoint_path / 'model-metadata.json').read_text())
trained_config = json.loads((checkpoint_path / 'model-config.json').read_text())
X_train = pd.read_csv(TRAINING / 'output/data/modeling/train/X.csv', index_col='row_index')
y_train = pd.read_csv(TRAINING / 'output/data/modeling/train/y.csv', index_col='row_index')
X_validation = pd.read_csv(TRAINING / 'output/data/modeling/validation/X.csv', index_col='row_index')
y_validation = pd.read_csv(TRAINING / 'output/data/modeling/validation/y.csv', index_col='row_index')
display(trained_config['parameters'])
display(metadata)
print('Transformed train and validation:', X_train.shape, X_validation.shape)
display(X_train.head())
display(y_train.head())

## Evaluate: score the held-out test split

- Call `evaluate.execute` with the saved checkpoint and Prepare's separate test CSV.
- Reload the fitted preprocessing and model without fitting either again.
- Inspect metrics, actual labels, predictions and the transformed test features.

In [ ]:
# Score held-out test rows with the saved model.
report = evaluate.execute(MODEL, checkpoint_path, EVALUATION,
                          input_path=split_dir / 'test/test.csv')

# Inspect evaluation metrics, predictions and the exact transformed test input.
evaluation_predictions = pd.read_csv(EVALUATION / 'predictions.csv', **CSV_OPTIONS)
X_test = pd.read_csv(EVALUATION / 'modeling/test/X.csv', index_col='row_index')
display(report)
display(evaluation_predictions.head())
display(X_test.head())

## QualityGate: inspect the registration decision

- Read the quality result written by Evaluate using the configured metric bounds.
- Stop this walkthrough if those bounds fail, matching the training pipeline's failure branch.
- Show whether the next managed step would register the model; make no registry call here.

No configured thresholds means no quality policy is enforced. A passing result in
that case is not evidence of model quality. Managed RegisterModel creates a
pending-approval package; it is a SageMaker service operation, not a Python entry point.

In [ ]:
quality = report['quality']
display(quality)
if not quality['configured']:
    print('No quality thresholds are configured for this model.')
if not quality['passed']:
    raise RuntimeError('QualityGate failed; inspect the evaluation report before continuing.')
print('QualityGate passed. Managed next step: RegisterModel with PendingManualApproval.')
print('No model was registered or approved by this notebook.')

## ValidatePackage: enter local-development inference mode

- Call the same package-validation entry point using its explicit local-development option.
- Save a record showing that registry approval was not checked for this local checkpoint.
- Keep this option local to the call; do not modify deployed pipeline settings.

This lets you debug a newly trained checkpoint before registration. Managed
inference normally requires an approved package with the exact image/checkpoint
pair; that check needs AWS access and is not exercised here.

In [ ]:
authorization = validate_package.execute(
    package_arn='', group='', image_uri='', checkpoint_uri=str(checkpoint_path),
    output_dir=AUTHORIZATION, require_approval=False,
)
display(authorization)
print('Local development only: registry approval has not been checked.')

## Preprocess: prepare the batch inference files

- Call `preprocess.execute` with the input CSV and saved checkpoint configuration.
- Keep the model's selected features and required identity fields, then write JSON-lines shards.
- Inspect the shard files and first batch dataframe before prediction.

This step arranges batch input. The saved training transformation is applied by
the predictor when each request is scored; it is not fitted again.

In [ ]:
# Run the batch-preprocessing entry point on the inference source.
shard_dir = preprocess.execute(
    MODEL, SOURCE, BATCH_INPUT, model_dir=checkpoint_path, shard_rows=BATCH_ROWS,
)

# Inspect the files and rows that will be sent to the serving application.
shard_paths = sorted(shard_dir.glob('*.jsonl'))
batch_frame = pd.read_json(shard_paths[0], lines=True, dtype={config['identifier']: str})
display(pd.DataFrame([{'file': path.name, 'bytes': path.stat().st_size} for path in shard_paths]))
print('Batch input columns:', list(batch_frame.columns))
display(batch_frame.head())

## Load the serving application and inspect its readiness

- Call `serve.execute` to load the real checkpoint and register the container's prediction routes.
- Create an in-process request client and inspect readiness and batch settings.
- Propagate handler exceptions into the notebook so the original traceback is available.

SageMaker CreateModel records an image/checkpoint pair; it does not run a model
function. Here we load that checkpoint directly. The request client calls the
actual Flask application without opening a port or starting Gunicorn. It is not
a managed CreateModel or Batch Transform job.

In [ ]:
# Load the same application that the batch container runs.
batch_application = serve.execute(MODEL, checkpoint_path)
batch_application.config['TESTING'] = True
batch_client = batch_application.test_client()

# Inspect the application routes without a server process.
readiness = batch_client.get('/ping')
if readiness.status_code != 200:
    raise RuntimeError(readiness.get_data(as_text=True))
print('Readiness:', readiness.status_code, readiness.get_data(as_text=True))
display(batch_client.get('/execution-parameters').get_json())

## Score the shards through the real prediction handler

- Send each prepared JSON-lines shard to the application's `/invocations` route.
- Use the loaded model and saved preprocessing to produce row IDs and scores.
- Save one `.out` file per shard and inspect the raw responses before postprocessing.

Requests run sequentially in this kernel. This exercises the serving code and
payload format, not container startup, network transport, managed payload limits
or parallel Batch Transform execution.

In [ ]:
# Pass each shard through the real handler and save its response.
BATCH_SCORES.mkdir(parents=True, exist_ok=False)
batch_requests = []
for shard_path in shard_paths:
    response = batch_client.post(
        '/invocations', data=shard_path.read_bytes(), content_type='application/jsonlines',
    )
    if response.status_code != 200:
        raise RuntimeError(f'{shard_path.name}: {response.get_data(as_text=True)}')
    score_path = BATCH_SCORES / (shard_path.name + '.out')
    score_path.write_text(response.get_data(as_text=True).rstrip('\n') + '\n')
    batch_requests.append({'input': shard_path.name, 'status': response.status_code,
                           'rows': len(response.data.splitlines()), 'output': score_path.name})

# Inspect the raw row IDs and scores consumed by Postprocess.
raw_scores = pd.read_json(
    next(iter(sorted(BATCH_SCORES.glob('*.out')))), lines=True, dtype={'row_id': str},
)
display(pd.DataFrame(batch_requests))
display(raw_scores.head())

## Postprocess: format the batch predictions

- Call `postprocess.execute` on the raw `.out` files and saved checkpoint configuration.
- Apply the selected use case's postprocessing to bounded chunks of scores and row IDs.
- Inspect the final prediction dataframe and the written summary.

In [ ]:
prediction_path = postprocess.execute(
    MODEL, BATCH_SCORES, BATCH_OUTPUT, model_dir=checkpoint_path, chunk_rows=BATCH_ROWS,
)
predictions = pd.read_csv(prediction_path, **CSV_OPTIONS)
batch_summary = json.loads((BATCH_OUTPUT / 'summary.json').read_text())
display(predictions.head())
display(batch_summary)
print('Batch predictions:', prediction_path)

## Direct Predict: compare the convenience path

- Call `predict.execute` on the same input and checkpoint, independently of the batch-file handoffs.
- Inspect the direct prediction dataframe alongside the batch predictions.
- Keep both results available for investigating custom preprocessing or postprocessing behavior.

Direct Predict was the shortcut used by the earlier notebook. It is not an extra
step in the managed batch definition. This walkthrough uses a small sample;
direct Predict and these inspection dataframes load their input in memory.

In [ ]:
direct_prediction_path = predict.execute(
    MODEL, SOURCE, checkpoint_path, RUN / 'predict/predictions.csv',
)
direct_predictions = pd.read_csv(direct_prediction_path, **CSV_OPTIONS)
display(direct_predictions.head())
print('Batch rows:', len(predictions), '| Direct rows:', len(direct_predictions))

## Review the handoffs and continue experimenting

- List the actual directories passed between the entry points in this run.
- Keep feature rankings, splits, transformed X/y, metrics and prediction dataframes in memory.
- Open the matching source files, edit the model logic, then restart the kernel and rerun.

Rerunning the setup creates a fresh run directory. To debug a failure, inspect the
traceback and notebook variables before restarting; IPython `%debug` can inspect
the failed call's frames.

Registration and approval have not occurred. Use **03C** to check real Local
containers and **03D** for the managed jobs, registry operations and execution logs.

In [ ]:
handoffs = {
    'Raw training input snapshot': PREPARED / 'raw',
    'Feature selection reports': PREPARED / 'selection',
    'Selected train / validation / test': split_dir,
    'Transformed training X/y': TRAINING / 'output/data/modeling',
    'Checkpoint and saved preprocessing': checkpoint_path,
    'Evaluation report and held-out predictions': EVALUATION,
    'Local package-validation record': AUTHORIZATION,
    'Batch input shards': shard_dir,
    'Raw prediction responses': BATCH_SCORES,
    'Formatted batch predictions': BATCH_OUTPUT,
}
display(pd.DataFrame([{'handoff': name, 'path under run folder': str(path.relative_to(RUN))}
                      for name, path in handoffs.items()]))
print('Run folder:', RUN)
print('Python entry-point walkthrough completed; no AWS resources were created.')